**Setup**

Import the required libraries and dataset. 

* Dataset: bitext/customer-support-llm-chatbot-training-dataset (Modified)
* Original: https://huggingface.co/datasets/bitext/customer-support-llm-chatbot-training-dataset

MODIFICATIONS (Educational):
- Added synthetic PII columns and injected PII into text
- Introduced data quality issues
- Purpose: Simulate realistic data preprocessing challenges

In [1]:
import pandas as pd
import numpy as np
import hashlib

# Load the dataset
df = pd.read_csv('./data/customer_support_clean.csv', on_bad_lines='skip', encoding='utf-8', sep=',', engine='python')

**Example**

Your dataset contains conversations that have already been routed using a three-tier strategy, indicated by columns `model_tier` and `model_used`. 

Examine the current dataset:

In [ ]:
print(f"Total conversations: {len(df):,}")
print(f"\nTiers:")
print(df['model_tier'].unique())
print(f"\nModels:")
print(df['model_used'].unique())
print(f"\nColumns:")
print(df.columns)

Total conversations: 26,872

Tiers:
<ArrowStringArray>
['tier2', 'tier3', 'tier1']
Length: 3, dtype: str

Models:
<ArrowStringArray>
['gpt-3.5-turbo', 'gpt-4-turbo', 'gpt-4', 'rule-based']
Length: 4, dtype: str


Before making improvements, you need to understand how much you're spending now.

1. Calculate the count and percentage of conversations in each tier (tier1, tier2, tier3); save as `tier_counts` and `tier_percentages`
2. Calculate the total cost for each tier
3. Calculate the average cost per conversation for each tier
4. Calculate what the cost would be if you used **only GPT-4** (Tier 3) for everything; use `gpt4_cost_per_conv` multiplied by the number of conversations for the `baseline_cost`
5. Calculate your current savings (total and percentage) compared to the all-GPT-4 baseline

In [ ]:
# Calculate distribution by tier
tier_counts = df['model_tier'].value_counts()
tier_percentages = (tier_counts / len(df)) * 100

# Calculate costs by tier
tier_costs = df.groupby('model_tier')['cost'].sum()
tier_avg_costs = df.groupby('model_tier')['cost'].mean()

# Calculate current total cost
current_total_cost = df['cost'].sum()

# Calculate baseline cost (all GPT-4)
# GPT-4 costs approximately $0.01 per 1K tokens, avg 180 tokens
gpt4_cost_per_conv = 0.018  # Average GPT-4 cost from data
baseline_cost = len(df) * gpt4_cost_per_conv

# Calculate savings
total_savings = baseline_cost - current_total_cost
savings_percentage = (total_savings / baseline_cost) * 100

# Display results
print("=== Current Routing Analysis ===")
print(f"\nTraffic Distribution:")
for tier in ['tier1', 'tier2', 'tier3']:
    if tier in tier_counts.index:
        count = tier_counts[tier]
        pct = tier_percentages[tier]
        print(f"  {tier}: {count:,} conversations ({pct:.1f}%)")

print(f"\nCost by Tier:")
for tier in ['tier1', 'tier2', 'tier3']:
    if tier in tier_costs.index:
        total = tier_costs[tier]
        avg = tier_avg_costs[tier]
        print(f"  {tier}: ${total:.2f} total (${avg:.4f} average)")

print(f"\n=== Cost Comparison ===")
print(f"Current total cost: ${current_total_cost:.2f}")
print(f"Baseline (all GPT-4): ${baseline_cost:.2f}")
print(f"Current savings: ${total_savings:.2f} ({savings_percentage:.1f}%)")


=== Current Routing Analysis ===

Traffic Distribution:
  tier1: 3 conversations (0.0%)
  tier2: 1,154 conversations (4.3%)
  tier3: 25,715 conversations (95.7%)

Cost by Tier:
  tier1: $0.00 total ($0.0000 average)
  tier2: $0.34 total ($0.0003 average)
  tier3: $90.81 total ($0.0035 average)

=== Cost Comparison ===
Current total cost: $91.16
Baseline (all GPT-4): $483.70
Current savings: $392.54 (81.2%)


#### Implement Caching System

Many users ask the same questions over and over. Instead of calling expensive APIs every time, you can save (cache) responses for repeated questions. This is one of the easiest ways to cut costs.

1. Create a `query_hash` column for each conversation's instruction using the created function `hash_query`
2. Identify the number of duplicate queries using the `query_hash` column; create a column `is_duplicate` in `df` to identify duplicate queries
3. Calculate the cache hit rate - what % of queries are repeats?
4. Calculate the cost savings from caching - add up the costs of all duplicate queries (these would be free with caching)
5. Calculate the total savings combining routing + caching

**How caching works:** The first time someone asks "What are your hours?", you call the API and cache the response. The next 100 people who ask get the cached response instantly for $0.

In [6]:
# Create query hash for each instruction
def hash_query(text):
    """Create MD5 hash of query for cache key"""
    if pd.isna(text):
        return None
    return hashlib.md5(str(text).lower().strip().encode()).hexdigest()

df['query_hash'] = df['instruction'].apply(hash_query)

# Count query occurrences
query_counts = df['query_hash'].value_counts()

# Identify duplicates (queries that appear more than once)
duplicate_hashes = query_counts[query_counts > 1].index
df['is_duplicate'] = df.duplicated(subset='query_hash')

# Calculate cache statistics
total_queries = len(df)
unique_queries = df['query_hash'].nunique()
duplicate_queries = df['is_duplicate'].sum()
cache_hit_rate = (duplicate_queries / total_queries) * 100

# Calculate cost savings from caching
# First occurrence of each query still needs API call (cost remains)
# All subsequent occurrences are cache hits (cost = $0)
first_occurrences = df.drop_duplicates(subset='query_hash')
cost_with_cache = first_occurrences['cost'].sum()
cache_savings = current_total_cost - cost_with_cache

# Calculate total savings (routing + caching)
total_optimized_savings = total_savings + cache_savings
total_savings_pct = (total_optimized_savings / baseline_cost) * 100

# Display results
print("=== Caching Analysis ===")
print(f"\nQuery Statistics:")
print(f"  Total queries: {total_queries:,}")
print(f"  Unique queries: {unique_queries:,}")
print(f"  Duplicate queries: {duplicate_queries:,}")
print(f"  Cache hit rate: {cache_hit_rate:.1f}%")

print(f"\nMost Common Queries:")
top_queries = query_counts.head(5)
for i, (query_hash, count) in enumerate(top_queries.items(), 1):
    sample_text = df[df['query_hash'] == query_hash]['instruction'].iloc[0]
    print(f"  {i}. Asked {count} times: {sample_text[:60]}...")

print(f"\n=== Cost Impact ===")
print(f"Current cost (with routing): ${current_total_cost:.2f}")
print(f"Cost savings from caching: ${cache_savings:.2f}")
print(f"Cost with routing + caching: ${cost_with_cache:.2f}")

print(f"\n=== Total Optimization ===")
print(f"Baseline (all GPT-4, no cache): ${baseline_cost:.2f}")
print(f"Optimized (routing + caching): ${cost_with_cache:.2f}")
print(f"Total savings: ${total_optimized_savings:.2f} ({total_savings_pct:.1f}%)")


=== Caching Analysis ===

Query Statistics:
  Total queries: 26,872
  Unique queries: 24,347
  Duplicate queries: 2,525
  Cache hit rate: 9.4%

Most Common Queries:
  1. Asked 21 times:     ...
  2. Asked 10 times: I expect a refund of {{Refund Amount}} dollars...
  3. Asked 10 times: i expect a reimbursement of {{Refund Amount}} dollars...
  4. Asked 10 times: I expect a rebate of {{Refund Amount}} dollars...
  5. Asked 10 times: i expect a rebate of {{Currency Symbol}}{{Refund Amount}}...

=== Cost Impact ===
Current cost (with routing): $91.16
Cost savings from caching: $8.96
Cost with routing + caching: $82.20

=== Total Optimization ===
Baseline (all GPT-4, no cache): $483.70
Optimized (routing + caching): $82.20
Total savings: $401.50 (83.0%)


Saving money only works if quality stays high. You need to ensure that your routing strategy doesn't decrease the effectiveness of problem-solving or customer satisfaction.

1. Calculate the resolution rate and average CSAT by tier and save in `quality_by_tier`
2. Print the metrics by tier - are customers equally satisfied?

**The key insight:** If Tier 2 resolves 70% of issues at `$0.004` each and Tier 3 resolves 75% at `$0.020` each, which is a better value? You need to consider cost AND quality together.

In [7]:
# Calculate quality metrics by tier
quality_by_tier = df.groupby('model_tier').agg(
    resolved=('resolved', 'mean'),
    csat_score=('csat_score', 'mean')
)

# Display results
print("=== Quality Metrics by Tier ===")
print(f"\n{'Tier':<10} {'Resolution':<12} {'CSAT':<8} ")
print("-" * 60)
for tier in ['tier1', 'tier2', 'tier3']:
    if tier in quality_by_tier.index:
        res = quality_by_tier.loc[tier, 'resolved']
        csat = quality_by_tier.loc[tier, 'csat_score']
        print(f"{tier:<10} {res:<12.1%} {csat:<8.2f}")
  

=== Quality Metrics by Tier ===

Tier       Resolution   CSAT     
------------------------------------------------------------
tier1      33.3%        3.13    
tier2      72.5%        4.03    
tier3      73.2%        4.05    


`tier1` has a low resolution rate _and_ CSAT score -- customers' questions aren't being answered, and they are unhappy about it! It may be time to reconsider the logic of the first-tier routing. However, your other two tiers have significantly higher resolutions and CSAT scores.